# Initialization    

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

# Reading from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

# Data transformations

In [0]:
df.display()

Analizing the table, we could transform the sls_order_dt, sls_ship_dt and sls_due_dt to proper data types. Also, we could rename the columns. Another thing that caught my eye is that sls_quantity appears to just be one in all rows, which is weird.

In [0]:
df.printSchema()

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Converting sls_due_dt, sls_ship_dt and sls_order_dt to Data type

In [0]:
df = df.withColumn("sls_order_dt",
                   F.try_to_date(F.col("sls_order_dt").cast(StringType()), "yyyyMMdd")
                   )


df = df.withColumn("sls_ship_dt",
                   F.try_to_date(F.col("sls_ship_dt").cast(StringType()), "yyyyMMdd")
                   
                   )


df = df.withColumn("sls_due_dt",
                   F.try_to_date(F.col("sls_due_dt").cast(StringType()), "yyyyMMdd"))

In [0]:
df.printSchema()

# Renaming Columns

In [0]:

RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
     


# Sanity check of dataframe

In [0]:
df.limit(10).display()

# Writing Silver Tabble

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

# Sanity check of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10